# Sayiir — Hello World

Minimal durable workflow example. [docs.sayiir.dev](https://docs.sayiir.dev)

In [ ]:
!pip install -q sayiir

## Define tasks and build a workflow

In [ ]:
from sayiir import task, Flow, run_workflow

@task
def fetch_user(user_id: int) -> dict:
    return {"id": user_id, "name": "Alice"}

@task
def send_email(user: dict) -> str:
    return f"Sent welcome to {user['name']}"

workflow = Flow("welcome").then(fetch_user).then(send_email).build()
result = run_workflow(workflow, 42)
print(result)

## Durable execution with checkpointing

In [ ]:
from sayiir import run_durable_workflow, InMemoryBackend

backend = InMemoryBackend()
status = run_durable_workflow(workflow, "run-001", 42, backend=backend)

print(f"Status: {status.status}")
print(f"Output: {status.output}")

## Parallel workflows (fork/join)

In [ ]:
@task
def double(x: int) -> int:
    return x * 2

@task
def add_ten(x: int) -> int:
    return x + 10

@task
def combine(results: dict) -> str:
    return f"doubled={results['double']}, plus_ten={results['add_ten']}"

parallel_wf = (
    Flow("parallel")
    .fork()
        .branch(double)
        .branch(add_ten)
    .join(combine)
    .build()
)

result = run_workflow(parallel_wf, 5)
print(result)

---

Learn more at [docs.sayiir.dev](https://docs.sayiir.dev) | [GitHub](https://github.com/sayiir/sayiir)